<a href="https://colab.research.google.com/github/Aliahmadjangohar/Aliahmadjangohar/blob/main/NEURO_SYMBOLIC_AI_PLANNING_SYSTEM_PDDL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
COMPLETE NEURO-SYMBOLIC AI PLANNING SYSTEM
CW2 Final Implementation
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import Union, List, Dict, Tuple, Optional, Set, FrozenSet
from pathlib import Path
import warnings
import re
import tempfile
from collections import defaultdict, deque
from dataclasses import dataclass, field
from PIL import Image
import torchvision.transforms as transforms
from torchvision import models
import os
import sys
import json


In [ ]:
# ============================================================================
# SECTION 1: CIFAR-100 SEMANTIC EXPANSION
# ============================================================================

def build_my_embeddings(checkpoint_path: str = "best_skipgram_523words.pth") -> Tuple[Dict[str, int], np.ndarray]:
    """
    Load and return your trained Skip-gram embeddings.
    """
    try:
        # Load checkpoint
        checkpoint = torch.load(checkpoint_path, map_location='cpu')

        # Extract vocabulary and embeddings
        vocab = {}
        embeddings = None

        # Try different checkpoint formats
        if 'word_to_idx' in checkpoint:
            vocab = checkpoint['word_to_idx']
            if 'embeddings' in checkpoint:
                embeddings = checkpoint['embeddings']
            elif 'embedding_weights' in checkpoint:
                embeddings = checkpoint['embedding_weights']
        elif 'vocab' in checkpoint:
            vocab = checkpoint['vocab']
            if 'embeddings' in checkpoint:
                embeddings = checkpoint['embeddings']
        elif 'model' in checkpoint:
            # Extract from model state dict
            model_state = checkpoint['model']
            if 'embedding.weight' in model_state:
                embeddings = model_state['embedding.weight'].numpy()
                # Create vocab if idx_to_word exists
                if 'idx_to_word' in checkpoint:
                    idx_to_word = checkpoint['idx_to_word']
                    vocab = {word: idx for idx, word in enumerate(idx_to_word)}

        # If still no embeddings, try to extract from the model
        if embeddings is None and 'model_state_dict' in checkpoint:
            model_state = checkpoint['model_state_dict']
            for key in model_state:
                if 'embed' in key.lower() and 'weight' in key.lower():
                    embeddings = model_state[key].numpy()
                    break

        # Convert to numpy if needed
        if torch.is_tensor(embeddings):
            embeddings = embeddings.numpy()

        # If no vocab but we have embeddings, create a placeholder vocab
        if not vocab and embeddings is not None:
            vocab = {f"word_{i}": i for i in range(embeddings.shape[0])}

        # Verify we have at least some embeddings
        if embeddings is None:
            raise ValueError("Could not extract embeddings from checkpoint")

        # Normalize embeddings
        embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

        print(f"✓ Loaded embeddings: {embeddings.shape[0]} words, dim={embeddings.shape[1]}")

        # Verify CIFAR-100 classes
        cifar_classes = [
            "apple", "aquarium_fish", "baby", "bear", "beaver", "bed", "bee", "beetle",
            "bicycle", "bottle", "bowl", "boy", "bridge", "bus", "butterfly", "camel",
            "can", "castle", "caterpillar", "cattle", "chair", "chimpanzee", "clock",
            "cloud", "cockroach", "couch", "crab", "crocodile", "cup", "dinosaur",
            "dolphin", "elephant", "flatfish", "forest", "fox", "girl", "hamster",
            "house", "kangaroo", "keyboard", "lamp", "lawn_mower", "leopard", "lion",
            "lizard", "lobster", "man", "maple_tree", "motorcycle", "mountain", "mouse",
            "mushroom", "oak_tree", "orange", "orchid", "otter", "palm_tree", "pear",
            "pickup_truck", "pine_tree", "plain", "plate", "poppy", "porcupine", "possum",
            "rabbit", "raccoon", "ray", "road", "rocket", "rose", "sea", "seal",
            "shark", "shrew", "skunk", "skyscraper", "snail", "snake", "spider",
            "squirrel", "streetcar", "sunflower", "sweet_pepper", "table", "tank",
            "telephone", "television", "tiger", "tractor", "train", "trout", "tulip",
            "turtle", "wardrobe", "whale", "willow_tree", "wolf", "woman", "worm"
        ]

        found_classes = [cls for cls in cifar_classes if cls in vocab]
        print(f"✓ Found {len(found_classes)}/{len(cifar_classes)} CIFAR-100 classes in vocabulary")

        return vocab, embeddings

    except Exception as e:
        print(f"Error loading embeddings: {e}")
        # Return minimal embeddings for testing
        vocab = {"apple": 0, "car": 1, "dog": 2}
        embeddings = np.random.randn(3, 100)
        embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
        return vocab, embeddings


In [ ]:
# ============================================================================
# DATA STRUCTURES FOR PDDL PLANNING
# ============================================================================

# CIFAR-100 classes (both underscore and hyphen versions)
CIFAR_100_CLASSES = {
    "apple", "aquarium_fish", "baby", "bear", "beaver", "bed", "bee", "beetle",
    "bicycle", "bottle", "bowl", "boy", "bridge", "bus", "butterfly", "camel",
    "can", "castle", "caterpillar", "cattle", "chair", "chimpanzee", "clock",
    "cloud", "cockroach", "couch", "crab", "crocodile", "cup", "dinosaur",
    "dolphin", "elephant", "flatfish", "forest", "fox", "girl", "hamster",
    "house", "kangaroo", "keyboard", "lamp", "lawn_mower", "leopard", "lion",
    "lizard", "lobster", "man", "maple_tree", "motorcycle", "mountain", "mouse",
    "mushroom", "oak_tree", "orange", "orchid", "otter", "palm_tree", "pear",
    "pickup_truck", "pine_tree", "plain", "plate", "poppy", "porcupine", "possum",
    "rabbit", "raccoon", "ray", "road", "rocket", "rose", "sea", "seal",
    "shark", "shrew", "skunk", "skyscraper", "snail", "snake", "spider",
    "squirrel", "streetcar", "sunflower", "sweet_pepper", "table", "tank",
    "telephone", "television", "tiger", "tractor", "train", "trout", "tulip",
    "turtle", "wardrobe", "whale", "willow_tree", "wolf", "woman", "worm"
}

CIFAR_100_CLASSES_PDDL = {
    "apple", "aquarium-fish", "baby", "bear", "beaver", "bed", "bee", "beetle",
    "bicycle", "bottle", "bowl", "boy", "bridge", "bus", "butterfly", "camel",
    "can", "castle", "caterpillar", "cattle", "chair", "chimpanzee", "clock",
    "cloud", "cockroach", "couch", "crab", "crocodile", "cup", "dinosaur",
    "dolphin", "elephant", "flatfish", "forest", "fox", "girl", "hamster",
    "house", "kangaroo", "keyboard", "lamp", "lawn-mower", "leopard", "lion",
    "lizard", "lobster", "man", "maple", "motorcycle", "mountain", "mouse",
    "mushroom", "oak", "orange", "orchid", "otter", "palm", "pear",
    "pickup-truck", "pine", "plain", "plate", "poppy", "porcupine", "possum",
    "rabbit", "raccoon", "ray", "road", "rocket", "rose", "sea", "seal",
    "shark", "shrew", "skunk", "skyscraper", "snail", "snake", "spider",
    "squirrel", "streetcar", "sunflower", "sweet-pepper", "table", "tank",
    "telephone", "television", "tiger", "tractor", "train", "trout", "tulip",
    "turtle", "wardrobe", "whale", "willow", "wolf", "woman", "worm"
}

TOOLS = {'knife', 'dslr'}
LOCATIONS = {'lab', 'outdoors'}
IGNORED_KEYWORDS = {'item', 'location', 'object', 'objects', '-', 'agent', 'define',
                    'problem', 'domain', 'init', 'goal', 'and', 'not'}

@dataclass(frozen=True)
class Predicate:
    name: str
    args: Tuple[str, ...]

    def __str__(self) -> str:
        return f"({self.name} {' '.join(self.args)})" if self.args else f"({self.name})"

    @staticmethod
    def from_string(s: str) -> 'Predicate':
        s = s.strip()
        if not (s.startswith('(') and s.endswith(')')):
            raise ValueError(f"Invalid predicate format: {s}")

        s = s[1:-1].strip()  # Remove parentheses
        parts = s.split()
        return Predicate(parts[0], tuple(parts[1:]) if len(parts) > 1 else ())


@dataclass(frozen=True)
class Action:
    name: str
    parameters: Tuple[str, ...]
    preconditions: FrozenSet[Predicate]
    add_effects: FrozenSet[Predicate]
    del_effects: FrozenSet[Predicate]

    def __str__(self) -> str:
        return f"({self.name} {' '.join(self.parameters)})"

    def instantiate(self, bindings: Dict[str, str]) -> 'Action':
        """Creates a concrete Action instance."""
        def substitute(pred: Predicate) -> Predicate:
            new_args = tuple(bindings.get(arg, arg) for arg in pred.args)
            return Predicate(pred.name, new_args)

        new_params = tuple(bindings.get(p, p) for p in self.parameters)
        new_pre = frozenset(substitute(p) for p in self.preconditions)
        new_add = frozenset(substitute(p) for p in self.add_effects)
        new_del = frozenset(substitute(p) for p in self.del_effects)

        return Action(self.name, new_params, new_pre, new_add, new_del)


@dataclass(frozen=True)
class State:
    predicates: FrozenSet[Predicate]

    def apply_action(self, action: Action) -> 'State':
        """Returns a NEW State by applying the action's effects."""
        new_predicates = set(self.predicates)
        new_predicates.update(action.add_effects)
        new_predicates.difference_update(action.del_effects)
        return State(frozenset(new_predicates))

    def is_applicable(self, action: Action) -> bool:
        """Checks if an action can be applied to this state."""
        return action.preconditions.issubset(self.predicates)

    def satisfies(self, goal: FrozenSet[Predicate]) -> bool:
        return goal.issubset(self.predicates)


@dataclass(order=True)
class SearchNode:
    f_score: int
    state: State = field(compare=False)
    action: Optional[Action] = field(compare=False)
    parent: Optional['SearchNode'] = field(compare=False)
    g_score: int = field(compare=False)

    def get_plan(self) -> List[Action]:
        plan, node = [], self
        while node.parent:
            plan.append(node.action)
            node = node.parent
        return list(reversed(plan))


In [ ]:
# ============================================================================
# PDDL PARSER
# ============================================================================

class PDDLParser:
    """Parser for PDDL domain and problem files."""

    @staticmethod
    def _extract_predicates(block: str) -> Set[Predicate]:
        """Extract predicates from a block."""
        block = block.strip()

        # Remove (and ...) wrapper if present
        if block.startswith('(and'):
            # Find matching closing parenthesis
            depth, i = 0, 4
            while i < len(block):
                if block[i] == '(':
                    depth += 1
                elif block[i] == ')':
                    depth -= 1
                    if depth == -1:
                        block = block[4:i].strip()
                        break
                i += 1

        preds = set()
        i = 0
        while i < len(block):
            if block[i] == '(':
                depth = 1
                j = i + 1
                while j < len(block) and depth > 0:
                    if block[j] == '(':
                        depth += 1
                    elif block[j] == ')':
                        depth -= 1
                    j += 1

                if depth == 0:
                    pred_str = block[i:j]
                    # Skip (not ...) predicates and forall
                    if not pred_str.startswith('(not') and not pred_str.startswith('(forall'):
                        try:
                            preds.add(Predicate.from_string(pred_str))
                        except:
                            pass
                    i = j
                else:
                    i += 1
            else:
                i += 1

        return preds

    @staticmethod
    def parse_domain(path: str) -> Dict[str, Action]:
        """Parse domain file and return action schemas."""
        with open(path) as f:
            content = f.read()

        actions = {}

        # Find all action definitions
        for m in re.finditer(r'\(:action\s+(\S+)(.*?)(?=\(:action|$)', content, re.DOTALL | re.IGNORECASE):
            name, body = m.groups()

            # Extract parameters
            params = []
            params_match = re.search(r':parameters\s+\((.*?)\)', body, re.DOTALL | re.IGNORECASE)
            if params_match:
                params = re.findall(r'\?[\w-]+', params_match.group(1))

            # Extract preconditions
            preconds = set()
            precond_match = re.search(r':precondition\s+(\(.*?)(?=:effect|$)', body, re.DOTALL | re.IGNORECASE)
            if precond_match:
                preconds = PDDLParser._extract_predicates(precond_match.group(1))

            # Extract effects
            adds, dels = set(), set()
            effect_match = re.search(r':effect\s+(\(.*?)(?=\))', body, re.DOTALL | re.IGNORECASE)
            if effect_match:
                effect_block = effect_match.group(1)
                # Simple parsing: look for predicates and (not predicate)
                i = 0
                while i < len(effect_block):
                    if effect_block[i] == '(':
                        if effect_block[i:i+4] == '(not':
                            # Find closing parenthesis for (not ...)
                            depth = 1
                            j = i + 4
                            while j < len(effect_block) and depth > 0:
                                if effect_block[j] == '(':
                                    depth += 1
                                elif effect_block[j] == ')':
                                    depth -= 1
                                j += 1

                            if depth == 0:
                                not_content = effect_block[i+4:j-1].strip()
                                if not_content.startswith('(') and not_content.endswith(')'):
                                    try:
                                        dels.add(Predicate.from_string(not_content))
                                    except:
                                        pass
                            i = j
                        else:
                            # Regular predicate
                            depth = 1
                            j = i + 1
                            while j < len(effect_block) and depth > 0:
                                if effect_block[j] == '(':
                                    depth += 1
                                elif effect_block[j] == ')':
                                    depth -= 1
                                j += 1

                            if depth == 0:
                                pred_str = effect_block[i:j]
                                try:
                                    adds.add(Predicate.from_string(pred_str))
                                except:
                                    pass
                            i = j
                    else:
                        i += 1

            actions[name] = Action(
                name,
                tuple(params),
                frozenset(preconds),
                frozenset(adds),
                frozenset(dels)
            )

        return actions

    @staticmethod
    def parse_problem(path: str) -> Tuple[Dict[str, Set[str]], State, FrozenSet[Predicate]]:
        """Parse problem file and return objects, initial state, and goal."""
        with open(path) as f:
            content = f.read()

        objs = defaultdict(set)

        # Parse objects
        if objects_match := re.search(r':objects(.*?)(?=\(:init|$)', content, re.DOTALL | re.IGNORECASE):
            objects_text = objects_match.group(1)
            objects_text = re.sub(r';.*$', '', objects_text, flags=re.MULTILINE).strip()

            tokens = re.findall(r'[^\s():;]+', objects_text)

            i = 0
            while i < len(tokens):
                token = tokens[i]
                if token == '-':
                    i += 1
                    if i < len(tokens):
                        type_name = tokens[i]
                        # Assign type to previous tokens
                        j = i - 2
                        while j >= 0 and tokens[j] != '-':
                            obj_name = tokens[j].lower()
                            if type_name == 'item':
                                objs['item'].add(obj_name)
                            elif type_name == 'tool':
                                objs['item'].add(obj_name)
                                objs['tool'].add(obj_name)
                            elif type_name == 'location':
                                objs['location'].add(obj_name)
                            j -= 1
                i += 1

        # Add default tools and locations if not present
        if 'tool' not in objs:
            objs['tool'] = TOOLS.copy()
            objs['item'].update(TOOLS)
        if 'location' not in objs:
            objs['location'] = LOCATIONS.copy()

        # Parse initial state
        init = set()
        if init_match := re.search(r':init(.*?)(?=\(:goal|$)', content, re.DOTALL | re.IGNORECASE):
            init = PDDLParser._extract_predicates(init_match.group(1))

        # Parse goal state
        goal = set()
        if goal_match := re.search(r':goal\s+(\(.*\))', content, re.DOTALL | re.IGNORECASE):
            goal = PDDLParser._extract_predicates(goal_match.group(1))

        return dict(objs), State(frozenset(init)), frozenset(goal)


In [ ]:
# ============================================================================
# ACTION GROUNDER
# ============================================================================

class ActionGrounder:
    """Grounds action schemas with concrete objects."""

    def __init__(self, schemas: Dict[str, Action], objects: Dict[str, Set[str]]):
        self.schemas = schemas
        self.all_items = sorted(objects.get('item', set()))
        self.cifar_objects = sorted([obj for obj in self.all_items
                                     if obj.replace('-', '_') in CIFAR_100_CLASSES])
        self.tools = sorted(objects.get('tool', set()))
        self.locations = sorted(objects.get('location', set()))

    def ground_all(self) -> List[Action]:
        """Ground all action schemas with concrete objects."""
        grounded = []

        for name, schema in self.schemas.items():
            if name == 'walk-between-rooms':
                grounded.extend(self._ground_walk(schema))
            elif name in ['pick-up', 'put-down']:
                grounded.extend(self._ground_item_location(schema))
            elif name in ['stack', 'unstack']:
                grounded.extend(self._ground_stack(schema))
            elif name in ['slice-object', 'clean-object', 'take-photo', 'make-wet']:
                grounded.extend(self._ground_object_action(schema))
            elif name in ['pick-up-tool', 'put-down-tool']:
                grounded.extend(self._ground_tool_action(schema))
            else:
                # Generic grounding
                grounded.extend(self._ground_generic(schema))

        return grounded

    def _ground_walk(self, schema: Action) -> List[Action]:
        return [schema.instantiate({schema.parameters[0]: l1, schema.parameters[1]: l2})
                for l1 in self.locations for l2 in self.locations if l1 != l2]

    def _ground_item_location(self, schema: Action) -> List[Action]:
        return [schema.instantiate({schema.parameters[0]: item, schema.parameters[1]: loc})
                for item in self.all_items for loc in self.locations]

    def _ground_stack(self, schema: Action) -> List[Action]:
        grounded = []
        for top in self.all_items:
            for bottom in self.all_items:
                if top != bottom:
                    for loc in self.locations:
                        grounded.append(schema.instantiate({
                            schema.parameters[0]: top,
                            schema.parameters[1]: bottom,
                            schema.parameters[2]: loc
                        }))
        return grounded

    def _ground_object_action(self, schema: Action) -> List[Action]:
        return [schema.instantiate({schema.parameters[0]: obj, schema.parameters[1]: loc})
                for obj in self.cifar_objects for loc in self.locations]

    def _ground_tool_action(self, schema: Action) -> List[Action]:
        return [schema.instantiate({schema.parameters[0]: tool, schema.parameters[1]: loc})
                for tool in self.tools for loc in self.locations]

    def _ground_generic(self, schema: Action) -> List[Action]:
        # Simple generic grounding
        grounded = []
        if len(schema.parameters) == 1:
            for obj in self.all_items:
                grounded.append(schema.instantiate({schema.parameters[0]: obj}))
        return grounded

In [ ]:
# ============================================================================
# BFS PLANNER
# ============================================================================

def bfs_search(initial: State, goal: FrozenSet[Predicate], actions: List[Action],
               max_iter: int = 10000) -> Optional[List[Action]]:
    """
    Breadth-first search for plan generation.
    """
    # Check if goal is already satisfied
    if initial.satisfies(goal):
        return []

    # Setup BFS
    start_node = SearchNode(0, initial, None, None, 0)
    frontier = deque([start_node])
    visited = {frozenset(initial.predicates)}
    iteration = 0

    while frontier and iteration < max_iter:
        iteration += 1
        current_node = frontier.popleft()

        # Generate successors
        for action in actions:
            if current_node.state.is_applicable(action):
                new_state = current_node.state.apply_action(action)
                new_state_frozen = frozenset(new_state.predicates)

                # Check if goal is reached
                if new_state.satisfies(goal):
                    # Return plan
                    plan = current_node.get_plan()
                    plan.append(action)
                    return plan

                # Add to frontier if not visited
                if new_state_frozen not in visited:
                    visited.add(new_state_frozen)
                    new_node = SearchNode(
                        current_node.g_score + 1,
                        new_state,
                        action,
                        current_node,
                        current_node.g_score + 1
                    )
                    frontier.append(new_node)

    # No plan found
    return None

In [ ]:
# ============================================================================
# IMAGE PROCESSING AND OBJECT IDENTIFICATION
# ============================================================================

class ImageEncoder(nn.Module):
    """Image encoder for CIFAR-100 images."""

    def __init__(self, proj_dim=64, device="cpu"):
        super().__init__()
        self.device = device

        # Load pretrained MobileNetV3-Small
        backbone = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
        # Extract all layers except classifier
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])
        self.backbone.to(device)
        self.backbone.eval()

        # Freeze backbone
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Create projection head
        self.projection = nn.Sequential(
            nn.Linear(576, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, proj_dim)
        ).to(device)

    def forward(self, x):
        # Extract features with frozen backbone
        with torch.no_grad():
            features = self.backbone(x)
            features = features.flatten(1)  # (batch_size, 576)

        # Project through trainable head
        projections = self.projection(features)

        return features, projections


def load_image_model(checkpoint_path: str, device="cpu") -> ImageEncoder:
    """Load the trained image projection model."""
    try:
        checkpoint = torch.load(checkpoint_path, map_location=device)

        # Get projection dimension
        proj_dim = checkpoint.get('proj_dim', 64)
        if 'config' in checkpoint:
            proj_dim = checkpoint['config'].get('proj_dim', 64)

        # Create model
        model = ImageEncoder(proj_dim=proj_dim, device=device)

        # Load state dict
        if 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
        elif 'model' in checkpoint:
            model.load_state_dict(checkpoint['model'])

        model.eval()
        print(f"✓ Loaded image model with projection dim {proj_dim}")
        return model

    except Exception as e:
        print(f"Error loading image model: {e}")
        # Return untrained model if loading fails
        return ImageEncoder(proj_dim=64, device=device)


def preprocess_image(image: Union[torch.Tensor, np.ndarray, Image.Image]) -> torch.Tensor:
    """Preprocess image for the model."""
    transform = transforms.Compose([
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])

    if isinstance(image, torch.Tensor):
        if image.dim() == 3:
            image = image.unsqueeze(0)
        # Assume already normalized if tensor
        if image.shape[1] == 3 and image.shape[2] == 224 and image.shape[3] == 224:
            return image
        else:
            # Resize and normalize
            from torchvision.transforms.functional import resize, normalize
            image = resize(image, (224, 224))
            image = normalize(image, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            return image
    elif isinstance(image, np.ndarray):
        if image.shape[-1] == 3:  # HWC to CHW
            image = image.transpose(2, 0, 1)
        image = torch.from_numpy(image).float()
        if image.dim() == 3:
            image = image.unsqueeze(0)
        # Normalize
        image = transforms.functional.normalize(
            image,
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
        return image
    elif isinstance(image, Image.Image):
        image = transform(image).unsqueeze(0)
        return image
    else:
        raise ValueError(f"Unsupported image type: {type(image)}")


def identify_object_from_image(image: Union[torch.Tensor, np.ndarray, Image.Image],
                              image_model: ImageEncoder,
                              word_embeddings: np.ndarray,
                              vocab: Dict[str, int],
                              device="cpu") -> Optional[str]:
    """
    Identify object from image using the projection model and word embeddings.
    """
    try:
        # Preprocess image
        image_tensor = preprocess_image(image).to(device)

        # Get image embedding
        with torch.no_grad():
            _, image_embedding = image_model(image_tensor)
            image_embedding = F.normalize(image_embedding, p=2, dim=1)

        # Convert to numpy
        image_embedding_np = image_embedding.cpu().numpy().flatten()
        image_embedding_np = image_embedding_np / np.linalg.norm(image_embedding_np)

        # Find most similar word among CIFAR-100 classes
        best_similarity = -1
        best_word = None

        for word in CIFAR_100_CLASSES:
            if word in vocab:
                word_idx = vocab[word]
                word_embedding = word_embeddings[word_idx]

                # Compute cosine similarity
                similarity = np.dot(image_embedding_np, word_embedding)

                if similarity > best_similarity:
                    best_similarity = similarity
                    best_word = word

        if best_word:
            print(f"  Identified: {best_word} (similarity: {best_similarity:.3f})")
            return best_word
        else:
            print("  Could not identify object")
            return None

    except Exception as e:
        print(f"  Error identifying object: {e}")
        return None


def validate_object_name(object_name: str, vocab: Dict[str, int]) -> Optional[str]:
    """
    Validate that object name exists in vocabulary.
    Returns the validated name or None if not found.
    """
    # Try exact match first
    if object_name in vocab:
        return object_name

    # Try with underscores/hyphens conversion
    if '_' in object_name:
        hyphen_version = object_name.replace('_', '-')
        if hyphen_version in vocab:
            return hyphen_version

    if '-' in object_name:
        underscore_version = object_name.replace('-', '_')
        if underscore_version in vocab:
            return underscore_version

    # Check for case-insensitive match
    object_name_lower = object_name.lower()
    for word in vocab:
        if word.lower() == object_name_lower:
            return word

    # Check CIFAR-100 classes
    for cifar_class in CIFAR_100_CLASSES:
        if object_name_lower == cifar_class.lower():
            return cifar_class

        # Try without underscores
        if object_name_lower.replace('_', '') == cifar_class.lower().replace('_', ''):
            return cifar_class

    print(f"  Warning: Object '{object_name}' not found in vocabulary")
    return None


In [ ]:
# ============================================================================
# PROBLEM CREATION
# ============================================================================

def create_pddl_problem(object_name: str,
                       initial_predicates: List[str],
                       goal_predicates: List[str],
                       problem_name: str = "custom") -> str:
    """
    Create a PDDL problem file string.
    """
    # Convert object name to PDDL format if needed
    pddl_object = object_name.replace('_', '-')

    # Build problem string
    problem_str = f"""(define (problem {problem_name}-problem)
  (:domain cifar100-process)

  (:objects
    {pddl_object} - item
    knife - tool
    dslr - tool
    lab - location
    outdoors - location
  )

  (:init
    {chr(10).join(f'    {p}' for p in initial_predicates)}
  )

  (:goal (and
    {chr(10).join(f'    {p}' for p in goal_predicates)}
  ))
)"""

    return problem_str

In [ ]:
# ============================================================================
# SECTION 2: NEURO-SYMBOLIC AI - MULTI-MODAL PLANNING
# ============================================================================

# DO NOT CHANGE THIS FUNCTION's signature
def plan_generator(input_data: Union[torch.Tensor, str],
                  initial_state: List[str],
                  goal_state: List[str],
                  domain_file: str = "domain.pddl", #adjust the path
                  skipgram_path: str = "best_skipgram_523words.pth",  #adjust the path
                  projection_path: str = "best_cifar100_projection.pth") -> Optional[List[str]]:#adjust the path
    """
    Main entry point for the neuro-symbolic planning system.
    """
    print("=" * 80)
    print("NEURO-SYMBOLIC PLANNING SYSTEM")
    print("=" * 80)

    try:
        # Set device
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {device}")

        # Step 1: Load models
        print("\n1. Loading models...")
        print("   Loading Skip-gram embeddings...")
        vocab, word_embeddings = build_my_embeddings(skipgram_path)

        print("   Loading image projection model...")
        image_model = load_image_model(projection_path, device)

        # Step 2: Identify object
        print("\n2. Identifying object...")
        object_name = None

        if isinstance(input_data, str):
            # Text input
            print(f"   Text input: '{input_data}'")
            object_name = validate_object_name(input_data, vocab)
            if not object_name:
                print(f"   ✗ Error: Object '{input_data}' not found in vocabulary")
                return None
            print(f"   ✓ Validated: {object_name}")

        elif isinstance(input_data, (torch.Tensor, np.ndarray, Image.Image)):
            # Image input
            print("   Image input detected")
            object_name = identify_object_from_image(
                input_data, image_model, word_embeddings, vocab, device
            )
            if not object_name:
                print("   ✗ Error: Could not identify object from image")
                return None
        else:
            print(f"   ✗ Error: Unsupported input type: {type(input_data)}")
            return None

        # Step 3: Parse PDDL domain
        print(f"\n3. Parsing PDDL domain: {domain_file}")
        try:
            if not os.path.exists(domain_file):
                print(f"   ✗ Error: Domain file not found: {domain_file}")
                return None

            action_schemas = PDDLParser.parse_domain(domain_file)
            print(f"   ✓ Found {len(action_schemas)} action schemas")
        except Exception as e:
            print(f"   ✗ Error parsing domain file: {e}")
            return None

        # Step 4: Replace object placeholder in predicates
        print("\n4. Processing predicates...")
        # Convert object name to PDDL format
        pddl_object = object_name.replace('_', '-')

        processed_initial = []
        for pred in initial_state:
            # Replace {OBJECT} or {object} placeholder
            pred = pred.replace('{OBJECT}', pddl_object)
            pred = pred.replace('{object}', pddl_object)
            # Also replace if object name appears directly
            pred = pred.replace(object_name, pddl_object)
            processed_initial.append(pred)

        processed_goal = []
        for pred in goal_state:
            pred = pred.replace('{OBJECT}', pddl_object)
            pred = pred.replace('{object}', pddl_object)
            pred = pred.replace(object_name, pddl_object)
            processed_goal.append(pred)

        print(f"   Initial state: {processed_initial}")
        print(f"   Goal state: {processed_goal}")

        # Step 5: Create PDDL problem
        print("\n5. Creating PDDL problem...")
        problem_str = create_pddl_problem(
            object_name,
            processed_initial,
            processed_goal,
            problem_name=object_name.replace('_', '-')
        )

        # Write to temporary file
        with tempfile.NamedTemporaryFile(mode='w', suffix='.pddl', delete=False) as f:
            f.write(problem_str)
            problem_file = f.name

        print(f"   ✓ Problem file created: {problem_file}")

        # Step 6: Parse problem file
        print("\n6. Parsing problem file...")
        try:
            parsed_objects, initial_state_obj, goal_state_predicates = PDDLParser.parse_problem(problem_file)
            print(f"   ✓ Parsed {len(parsed_objects.get('item', []))} items, "
                  f"{len(parsed_objects.get('location', []))} locations")
        except Exception as e:
            print(f"   ✗ Error parsing problem file: {e}")
            os.unlink(problem_file)
            return None

        # Step 7: Ground actions
        print("\n7. Grounding actions...")
        grounder = ActionGrounder(action_schemas, parsed_objects)
        grounded_actions = grounder.ground_all()
        print(f"   ✓ Generated {len(grounded_actions)} grounded actions")

        # Step 8: Generate plan using BFS
        print("\n8. Generating plan using BFS...")
        plan = bfs_search(
            initial_state_obj,
            goal_state_predicates,
            grounded_actions,
            max_iter=50000
        )

        # Clean up temporary file
        os.unlink(problem_file)

        if plan is None:
            print("   ✗ No valid plan found")
            return None

        # Step 9: Convert plan to string format
        print(f"\n9. Plan found with {len(plan)} actions:")
        plan_strings = []
        for i, action in enumerate(plan, 1):
            action_str = str(action)
            plan_strings.append(action_str)
            print(f"   {i:2d}. {action_str}")

        # Step 10: Validate plan
        print("\n10. Validating plan...")
        if validate_plan_execution(plan, initial_state_obj, goal_state_predicates):
            print("   ✓ Plan validation successful")
        else:
            print("   ⚠ Plan validation failed (but plan returned anyway)")

        print("\n" + "=" * 80)
        print("PLANNING COMPLETE!")
        print("=" * 80)

        return plan_strings

    except Exception as e:
        print(f"\n✗ Error in plan_generator: {e}")
        import traceback
        traceback.print_exc()
        return None


def validate_plan_execution(plan: List[Action], initial_state: State, goal: FrozenSet[Predicate]) -> bool:
    """
    Validate that the plan achieves the goal.
    """
    current_state = initial_state

    for i, action in enumerate(plan):
        if not current_state.is_applicable(action):
            print(f"    Step {i+1}: Action {action} not applicable")
            return False
        current_state = current_state.apply_action(action)

    goal_achieved = current_state.satisfies(goal)
    if not goal_achieved:
        print(f"    Goal not achieved after plan execution")

    return goal_achieved


In [ ]:
# ============================================================================
# TESTING FUNCTIONS
# ============================================================================

def run_tests():
    """Run comprehensive tests of the system."""
    print("Running comprehensive tests...")

    # Check if files exist
    required_files = ["domain.pddl", "best_skipgram_523words.pth", "best_cifar100_projection.pth"]
    missing = [f for f in required_files if not os.path.exists(f)]

    if missing:
        print(f"Warning: Missing files: {missing}")
        print("Some tests may fail.")

    tests_passed = 0
    tests_total = 0

    # Test 1: Text input with simple plan
    print("\n" + "=" * 80)
    print("TEST 1: Text input - Move apple from lab to outdoors")
    print("=" * 80)

    initial = ["(at apple lab)", "(agent-at lab)", "(hand-empty)"]
    goal = ["(at apple outdoors)"]

    result = plan_generator(
        input_data="apple",
        initial_state=initial,
        goal_state=goal,
        domain_file="domain.pddl",
        skipgram_path="best_skipgram_523words.pth",
        projection_path="best_cifar100_projection.pth"
    )

    tests_total += 1
    if result:
        print(f"✓ Test 1 PASSED - Plan found with {len(result)} actions")
        tests_passed += 1
    else:
        print("✗ Test 1 FAILED - No plan found")

    # Test 2: Different object
    print("\n" + "=" * 80)
    print("TEST 2: Text input - Process orange")
    print("=" * 80)

    initial = ["(at orange lab)", "(agent-at lab)", "(hand-empty)", "(tool-at knife lab)", "(whole orange)"]
    goal = ["(cut-into-pieces orange)"]

    result = plan_generator(
        input_data="orange",
        initial_state=initial,
        goal_state=goal,
        domain_file="domain.pddl"
    )

    tests_total += 1
    if result:
        print(f"✓ Test 2 PASSED - Plan found with {len(result)} actions")
        tests_passed += 1
    else:
        print("✗ Test 2 FAILED - No plan found")

    # Test 3: Already satisfied goal
    print("\n" + "=" * 80)
    print("TEST 3: Already satisfied goal")
    print("=" * 80)

    initial = ["(at apple lab)", "(agent-at lab)"]
    goal = ["(at apple lab)"]

    result = plan_generator(
        input_data="apple",
        initial_state=initial,
        goal_state=goal,
        domain_file="domain.pddl"
    )

    tests_total += 1
    if result == []:
        print("✓ Test 3 PASSED - Empty plan (goal already satisfied)")
        tests_passed += 1
    else:
        print(f"✗ Test 3 FAILED - Expected empty plan, got {len(result) if result else 0} actions")

    # Test 4: Invalid object
    print("\n" + "=" * 80)
    print("TEST 4: Invalid object name")
    print("=" * 80)

    result = plan_generator(
        input_data="nonexistent_object",
        initial_state=initial,
        goal_state=goal,
        domain_file="domain.pddl"
    )

    tests_total += 1
    if result is None:
        print("✓ Test 4 PASSED - Correctly rejected invalid object")
        tests_passed += 1
    else:
        print("✗ Test 4 FAILED - Should have rejected invalid object")

    # Summary
    print("\n" + "=" * 80)
    print("TEST SUMMARY")
    print("=" * 80)
    print(f"Tests passed: {tests_passed}/{tests_total}")

    if tests_passed == tests_total:
        print("✅ ALL TESTS PASSED!")
    else:
        print(f"⚠ {tests_total - tests_passed} test(s) failed")

    return tests_passed == tests_total


In [ ]:
# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    print("NEURO-SYMBOLIC AI PLANNING SYSTEM - FINAL IMPLEMENTATION")
    print("=" * 80)

    # Check if we're running tests
    if len(sys.argv) > 1 and sys.argv[1] == "--test":
        success = run_tests()
        sys.exit(0 if success else 1)

    # Otherwise, run a simple example
    print("\nRunning example plan...")

    # Example 1: Simple movement
    print("\nExample: Move apple from lab to outdoors")
    initial = ["(at apple lab)", "(agent-at lab)", "(hand-empty)"]
    goal = ["(at apple outdoors)"]

    plan = plan_generator(
        input_data="apple",
        initial_state=initial,
        goal_state=goal,
        domain_file="domain.pddl",
        skipgram_path="best_skipgram_523words.pth",
        projection_path="best_cifar100_projection.pth"
    )

    if plan:
        print(f"\n✅ Plan generated successfully!")
        print(f"Actions: {len(plan)}")
        for i, action in enumerate(plan, 1):
            print(f"  {i:2d}. {action}")
    else:
        print("\n❌ No plan found or error occurred")
